In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from huggingface_hub import login
login(new_session=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
import pandas as pd
import argparse
import sys
import os
from tqdm import tqdm
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer
import torch

def load_llm_model(model_name):
    """
    Loads the LLM model using the transformers library.
    """
    print(f"Loading model {model_name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

    return model, tokenizer


import json

def generate_query_plan(query, model, tokenizer):
    # 1. FULL, VALID EXAMPLE (Do not truncate this!)
    example_sql = """SELECT u.DisplayName, COUNT(p.Id) AS PostCount
FROM Users u
JOIN Posts p ON u.Id = p.OwnerUserId
WHERE u.Reputation > 1000
GROUP BY u.DisplayName
ORDER BY PostCount DESC
LIMIT 10;"""

    # A real, complete plan for the simple query above
    example_plan = """[
  {
    "Plan": {
      "Node Type": "Limit",
      "Startup Cost": 51.64,
      "Total Cost": 51.67,
      "Plan Rows": 10,
      "Plans": [
        {
          "Node Type": "Sort",
          "Parent Relationship": "Outer",
          "Sort Key": ["(count(p.id)) DESC"],
          "Plans": [
            {
              "Node Type": "Aggregate",
              "Strategy": "Sorted",
              "Parent Relationship": "Outer",
              "Group Key": ["u.displayname"],
              "Plans": [
                {
                  "Node Type": "Merge Join",
                  "Parent Relationship": "Outer",
                  "Join Type": "Inner",
                  "Merge Cond": "(u.id = p.owneruserid)",
                  "Plans": [
                    {
                      "Node Type": "Sort",
                      "Parent Relationship": "Outer",
                      "Sort Key": ["u.id"],
                      "Plans": [
                        {
                          "Node Type": "Seq Scan",
                          "Parent Relationship": "Outer",
                          "Relation Name": "users",
                          "Alias": "u",
                          "Filter": "(reputation > 1000)"
                        }
                      ]
                    },
                    {
                      "Node Type": "Sort",
                      "Parent Relationship": "Inner",
                      "Sort Key": ["p.owneruserid"],
                      "Plans": [
                        {
                          "Node Type": "Seq Scan",
                          "Parent Relationship": "Outer",
                          "Relation Name": "posts",
                          "Alias": "p"
                        }
                      ]
                    }
                  ]
                }
              ]
            }
          ]
        }
      ]
    }
  }
]"""

    prompt = f"""You are a PostgreSQL query optimizer. Given a SQL query, output ONLY the valid JSON execution plan.

Example:
Input SQL:
{example_sql}

Output JSON:
{example_plan}

Now your turn.
Input SQL:
{query}

Output JSON:
"""

    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # 2. Deterministic Generation
        outputs = model.generate(
            **inputs,
            max_new_tokens=4096,      # Enough space for a full plan
            do_sample=False,          # Deterministic
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # 3. Robust Extraction (CRITICAL STEP)
        # Remove prompt
        completion = generated_text[len(prompt):].strip()

        # Slice exactly from first [ to last ]
        start_idx = completion.find('[')
        end_idx = completion.rfind(']')

        if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
            completion = completion[start_idx:end_idx+1]
        else:
             # Fallback if the model didn't output a clean list
             return "Error: Model did not generate valid JSON list."

        return completion

    except Exception as e:
        return f"Error generating plan: {e}"




def process_csv(input_file, output_file, query_column, model_name, limit=None):
    """
    Reads a CSV, generates query plans, and saves the result.
    Automatically resumes from where it left off by checking the output file.
    """
    if not os.path.exists(input_file):
        print(f"Error: Input file '{input_file}' not found.")
        return

    print(f"Reading input CSV from {input_file}...")
    df = pd.read_csv(input_file)

    if query_column not in df.columns:
        print(
            f"Error: Column '{query_column}' not found in CSV. Available columns: {list(df.columns)}")
        return

    # Apply limit to the input dataframe if specified
    if limit:
        print(f"Limiting input to first {limit} rows.")
        df = df.head(limit)

    # If 'plan_json' already exists in input, rename it to avoid confusion
    # This ensures we don't mistakenly think we are done if the input file has this column
    if 'plan_json' in df.columns:
        print("Renaming existing 'plan_json' column in input to 'original_plan_json'...")
        df.rename(columns={'plan_json': 'original_plan_json'}, inplace=True)

    # Initialize 'plan_json' column
    df['plan_json'] = None

    # Check for existing output file to resume progress
    if os.path.exists(output_file):
        print(
            f"Found existing output file '{output_file}'. Merging progress...")
        try:
            df_out = pd.read_csv(output_file)
            if 'plan_json' in df_out.columns:
                # Update the main dataframe with values from the output file
                # We align on index. This assumes the index hasn't changed between runs.
                # combine_first fills nulls in df with values from df_out
                df['plan_json'] = df_out['plan_json'].combine_first(
                    df['plan_json'])
                print("Merged existing plans from output file.")
            else:
                print("Output file exists but has no 'plan_json'. Ignoring.")
        except Exception as e:
            print(f"Error reading output file: {e}. Starting fresh.")

    # Find the first row that needs processing
    # We look for rows where plan_json is null
    missing_mask = df['plan_json'].isna()

    if not missing_mask.any():
        print("All rows already have plans. Processing complete.")
        return

    start_index = missing_mask.idxmax()
    total_rows = len(df)
    rows_to_process = missing_mask.sum()

    print(
        f"Total rows: {total_rows}. Already done: {total_rows - rows_to_process}. Remaining: {rows_to_process}.")
    print(f"Resuming from index {start_index}...")

    # Load model only if there is work to do
    model, tokenizer = load_llm_model(model_name)

    save_interval = 100
    print(f"Processing queries and saving every {save_interval} rows...")

    # Iterate only through the rows that need processing
    # We use the mask to filter, but we need to keep the original index to update df correctly
    for index, row in tqdm(df[missing_mask].iterrows(), total=rows_to_process):
        query = row[query_column]
        plan = generate_query_plan(query, model, tokenizer)
        df.at[index, 'plan_json'] = plan

        # Save periodically
        # We check the number of processed items, not the index
        if (index + 1) % save_interval == 0:
            df.to_csv(output_file, index=False)

    print(f"Saving final results to {output_file}...")
    df.to_csv(output_file, index=False)
    print("Processing complete.")




# Define the arguments directly for notebook execution
input_csv_file = "/content/drive/MyDrive/stackoverflow_n18147.csv"
output_csv_file = "/content/drive/MyDrive/output_with_plans_json_new.csv"
column_name = "sql_text"
llm_model_name = "abharadwaj123/llama3-sql2plan"
row_limit = 1 # Set to an integer like 100 to limit rows, or None for no limit

# Call the main processing function
process_csv(input_csv_file, output_csv_file, column_name, llm_model_name, row_limit)

Reading input CSV from /content/drive/MyDrive/stackoverflow_n18147.csv...
Limiting input to first 1 rows.
Renaming existing 'plan_json' column in input to 'original_plan_json'...
Total rows: 1. Already done: 0. Remaining: 1.
Resuming from index 0...
Loading model abharadwaj123/llama3-sql2plan...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Processing queries and saving every 100 rows...


100%|██████████| 1/1 [02:08<00:00, 128.79s/it]

Saving final results to /content/drive/MyDrive/output_with_plans_json_new.csv...
Processing complete.
